<a href="https://colab.research.google.com/github/asoknaren/AIPlayground/blob/main/001_tokenizer_microsoft_deberta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import AutoModel, AutoTokenizer
import torch
import time

In [2]:
MODEL_NAME = "microsoft/deberta-v3-xsmall"
TOKENIZER_NAME = "microsoft/deberta-base"
TARGET_WORDS = 200

In [3]:
def build_prompt() -> str:
    return (
        "Write a sincere professional apology email in about 200 words. "
        "Context: I missed an important client deadline due to poor planning. "
        "The email should acknowledge responsibility, explain briefly without excuses, "
        "and include a clear corrective action plan and request to rebuild trust."
    )


In [4]:
prompt = build_prompt()

In [5]:
def initialize_model_and_tokenizer(model_name: str):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
    model = AutoModel.from_pretrained(model_name, dtype=torch.bfloat16)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    if device == "cpu":
        cpu_threads = max(1, (torch.get_num_threads() or 1) - 1)
        torch.set_num_threads(cpu_threads)
        print(f"Using CPU threads: {cpu_threads}")

    return tokenizer, model, device

In [6]:
tokenizer, model, device = initialize_model_and_tokenizer(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

Using CPU threads: 1


model.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

## Show Tokens Helper Function

To better visualize the tokenization process, here's a helper function that displays tokens with different background colors.

In [7]:
colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence):
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

In [8]:
show_tokens(prompt)

[CLS] Write  a  sincere  professional  apology  email  in  about  200  words .  Context :  I  missed  an  important  client  deadline  due  to  poor  planning .  The  email  should  acknowledge  responsibility ,  explain  briefly  without  excuses ,  and  include  a  clear  corrective  action  plan  and  request  to  rebuild  trust . [SEP] 

In [9]:
def tokenize_and_display_prompt_info(tokenizer, prompt: str, device: str):
    # Use the tokenizer directly for encoding, as chat_template is not applicable for this model
    model_inputs = tokenizer(prompt, return_tensors="pt")
    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}
    input_ids = model_inputs["input_ids"]
    token_count = input_ids.shape[1]

    print(f"Prompt token count: {token_count}")
    print(f"Equivalent tokenized words: {tokenizer.decode(input_ids[0], skip_special_tokens=True)}")

    print("\nToken to Word Mapping:")
    for token_id in input_ids[0]:
        token_word = tokenizer.decode(token_id)
        print(f"Token ID: {token_id.item()}, Word: '{token_word}'")

    return model_inputs, token_count

In [10]:
model_inputs, token_count = tokenize_and_display_prompt_info(tokenizer, prompt, device)

Prompt token count: 50
Equivalent tokenized words: Write a sincere professional apology email in about 200 words. Context: I missed an important client deadline due to poor planning. The email should acknowledge responsibility, explain briefly without excuses, and include a clear corrective action plan and request to rebuild trust.

Token to Word Mapping:
Token ID: 1, Word: '[CLS]'
Token ID: 45714, Word: 'Write'
Token ID: 10, Word: ' a'
Token ID: 19255, Word: ' sincere'
Token ID: 2038, Word: ' professional'
Token ID: 9664, Word: ' apology'
Token ID: 1047, Word: ' email'
Token ID: 11, Word: ' in'
Token ID: 59, Word: ' about'
Token ID: 1878, Word: ' 200'
Token ID: 1617, Word: ' words'
Token ID: 4, Word: '.'
Token ID: 43885, Word: ' Context'
Token ID: 35, Word: ':'
Token ID: 38, Word: ' I'
Token ID: 2039, Word: ' missed'
Token ID: 41, Word: ' an'
Token ID: 505, Word: ' important'
Token ID: 3653, Word: ' client'
Token ID: 4267, Word: ' deadline'
Token ID: 528, Word: ' due'
Token ID: 7, Wor

The `microsoft/deberta-v3-xsmall` model is an **encoder-only** transformer model. Unlike models like `phi-3-mini` (which is a decoder-only model used for text generation), encoder-only models are primarily designed for understanding and encoding text into rich numerical representations (embeddings).

Here's what you can typically do with an encoder-only model like DeBERTa:

*   **Text Classification**: Classify text into categories (e.g., sentiment analysis, spam detection, topic classification).
*   **Named Entity Recognition (NER)**: Identify and classify named entities in text (e.g., person names, organizations, locations).
*   **Question Answering (Extractive)**: Given a question and a context paragraph, find the exact span of text in the context that answers the question.
*   **Semantic Search**: Find documents or passages that are semantically similar to a query.
*   **Text Similarity/Paraphrase Detection**: Determine how similar two pieces of text are.

The model's output provides contextual embeddings for each token in the input. These embeddings capture the meaning of words in the context of the surrounding words. You can then use these embeddings as features for a downstream task (often by adding a simple classification head on top of the encoder).

In [11]:
# Get the model's output (contextual embeddings)
with torch.no_grad():
    model_output = model(**model_inputs)

# The last_hidden_state contains the contextual embeddings for each token
# The shape is (batch_size, sequence_length, hidden_size)
last_hidden_state = model_output.last_hidden_state

print(f"Shape of last_hidden_state: {last_hidden_state.shape}")

# To get a single sentence embedding, you can typically average the token embeddings
# or use the embedding of the [CLS] token (the first token).

# Option 1: Average pooling (mean of all token embeddings)
sentence_embedding_avg = torch.mean(last_hidden_state, dim=1)
print(f"Shape of averaged sentence embedding: {sentence_embedding_avg.shape}")

# Option 2: Using the [CLS] token embedding (first token)
# This is common for classification tasks where the [CLS] token is specifically trained to represent the whole sequence.
sentence_embedding_cls = last_hidden_state[:, 0, :]
print(f"Shape of [CLS] token embedding: {sentence_embedding_cls.shape}")

print("\nFirst 5 values of the [CLS] token embedding:")
print(sentence_embedding_cls[0, :5])

print("\nThese embeddings can then be used as input features for a task-specific head (e.g., a linear layer for classification).")

Shape of last_hidden_state: torch.Size([1, 50, 384])
Shape of averaged sentence embedding: torch.Size([1, 384])
Shape of [CLS] token embedding: torch.Size([1, 384])

First 5 values of the [CLS] token embedding:
tensor([-3.5000, -0.1357, -0.0308, -0.0605, -0.0216], dtype=torch.bfloat16)

These embeddings can then be used as input features for a task-specific head (e.g., a linear layer for classification).
